# nbskill

`nbskill` is notebook-aware tooling for agents and maintainers working in nbdev projects. It provides focused context, structured edits, safe execution, and review through Python, the CLI, and MCP.

In [ ]:
#| hide
from contextlib import redirect_stdout
from io import StringIO

from nbskill.foundation import cell_source, parse_cells
from nbskill.graph import notebook_knowledge_graph_data
from nbskill.read import context

## Why it exists

In nbdev, the notebook is the source of truth. It holds the implementation, its explanation, examples, outputs, and metadata. Raw JSON edits are easy to misplace, and changes to generated Python are overwritten. `nbskill` makes notebook work deliberate and reviewable.

## What it does

Use `nbskill` to find the relevant project, notebook, chapter, cell, or symbol; edit the right cells; run focused checks in project context; and review the resulting diff and style. The MCP server exposes the same workflow to coding agents.

## What it offers

| Need | Tooling |
| --- | --- |
| Find the right context | [context](https://morscerta-crypto.github.io/nbskill/cli.html#context) reads a project, notebook, chapter, cell id, or symbol without raw JSON noise. |
| Make focused edits | [edit_notebook](https://morscerta-crypto.github.io/nbskill/edit.html#edit_notebook), [write_nb](https://morscerta-crypto.github.io/nbskill/cli.html#write_nb), and [update_cell](https://morscerta-crypto.github.io/nbskill/cli.html#update_cell) preserve cell structure and clear stale outputs. |
| Verify behavior | [exec_nb](https://morscerta-crypto.github.io/nbskill/cli.html#exec_nb) runs the notebook in project context, with a check-only mode for review loops. |
| Review changes | [diff_nb](https://morscerta-crypto.github.io/nbskill/cli.html#diff_nb), [style_check](https://morscerta-crypto.github.io/nbskill/cli.html#style_check), and `doctor` focus on code cells, notebook hygiene, and tool diagnostics. |
| Move code into nbdev | [convert](https://morscerta-crypto.github.io/nbskill/cli.html#convert) turns Python files or packages into nbdev notebooks. |
| Serve agents | [nbskill_mcp](https://morscerta-crypto.github.io/nbskill/cli.html#nbskill_mcp) exposes the same workflow as MCP tools. |

## Parse notebook cells

Notebook text uses `%%markdown`, `%%code`, and `---` as a compact cell format. `parse_cells` turns that text into notebook cells while preserving each cell's type and source.

In [10]:
cell_text = "\n".join(["%%markdown", "## Demo", "---", "%%code", "value = 42"])
parsed_cells = parse_cells(cell_text)
[(cell.cell_type, cell_source(cell).splitlines()[0]) for cell in parsed_cells]

[('markdown', '## Demo'), ('code', 'value = 42')]

## Read a notebook

Start with the smallest useful view. `context` resolves a project, notebook, chapter, cell, or symbol without exposing notebook JSON.

```bash
context --target nbs/02_write.ipynb --overview
context --target nbs/02_write.ipynb#write_nb
```

The first call explains the notebook. The second follows the public `write_nb` symbol to its owning cell and related uses.

In [9]:
with redirect_stdout(StringIO()): notebook_context = context("02_write.ipynb", overview=True)
print("\n".join([x['source'] for x in notebook_context["selection"]["markdown"]]))

## Writing and changing

Cell creation, targeted updates, documentation insertion, and examples.
Writing notebooks safely is harder than appending text to a file. A notebook edit needs to preserve cell ids, clear stale outputs, validate Python when possible, export through nbdev automatically, and avoid overwriting the wrong cell.

`write_nb` inserts cells; `replace_str` replaces literal text across notebook cell sources.
There are two writing modes because notebook edits have two different shapes. Use cell-block writes when adding or replacing structured cells, and use literal replacement only for exact renames across existing cells. Both paths stamp nbskill metadata and export affected notebooks automatically when they have an nbdev export target.


## MCP and CLI

Run `nbskill_mcp` when an agent client supports MCP. It exposes the same read, edit, execution, review, reference, and graph operations as structured tools. Use the CLI when working in a shell. The [CLI reference notebook](https://github.com/MorsCerta-crypto/nbskill/blob/main/nbs/13_cli.ipynb) lists every command.

Install the Codex skill and register the local MCP server with:

```bash
uv run install_nbskill --target codex
codex mcp add nbskill -- nbskill_mcp
```

Start the server directly when testing it:

```bash
uv run nbskill_mcp
uv run context --target nbs/02_write.ipynb
```

An MCP client starts with a health check, then asks for focused context:

```python
mcp__nbskill__.healthcheck()
mcp__nbskill__.context(target="nbs/02_write.ipynb#write_nb", overview=True)
```

## References

References are a local implementation knowledgebase, not code to copy. Discover or register repositories with `reference_discover` or `reference_add`, ingest them with `reference_ingest`, then query them with `reference_query` before adding a nontrivial helper. Use `reference_propose` when a match is worth turning into review material. Results combine structured filters, hybrid BM25/vector search, dependency status, and optional branch context.

```python
mcp__nbskill__.reference(
    action="query",
    query="AST definition helper",
    top_k=5,
    include_branch=True,
)
```

## Graph structure

The local graph records notebooks, cells, headings, symbols, and imports. Its edges record containment, definitions, calls, imports, documentation order, and related symbols. Call and import edges carry evidence separately from heuristic similarity edges.

Ask for project context with the graph attached when you need to trace a symbol through the notebook structure:

```python
mcp__nbskill__.context(target="project", scope="nbs", include_graph=True)
```

In [4]:
graph = notebook_knowledge_graph_data("nbs")
dict(nodes=len(graph["nodes"]), edges=len(graph["edges"]), node_types=sorted({node["type"] for node in graph["nodes"]}), edge_types=sorted({edge["type"] for edge in graph["edges"]}))

{'nodes': 3588,
 'edges': 13426,
 'node_types': ['cell', 'heading', 'import', 'notebook', 'symbol'],
 'edge_types': ['calls',
  'contains',
  'defines',
  'documents',
  'imports',
  'precedes',
  'similar_to']}